In [3]:
import os

# 1. Set the Kaggle environment variable using your new token
os.environ['KAGGLE_API_TOKEN'] = "KGAT_7404f046459c3f58a5b39ed3b3312654"

# 2. Download the Chest X-Ray (Pneumonia) dataset directly to Colab
print("Downloading the dataset from Kaggle...")
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

# 3. Unzip the downloaded file silently (-q) into a new folder
print("Unzipping the dataset...")
!unzip -q chest-xray-pneumonia.zip -d chest_xray_data
print("\nSuccess! Dataset downloaded and extracted.")

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [00:14<00:00, 170MB/s]

Unzipping the dataset...

Success! Dataset downloaded and extracted.


In [4]:
import tensorflow as tf
import os

# 1. Define where the data was extracted
base_dir = 'chest_xray_data/chest_xray'
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test') # Using test folder for validation

IMG_SIZE = (150, 150)
BATCH_SIZE = 32

# 2. Load the X-ray images into TensorFlow datasets
print("Loading training data...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

print("\nLoading validation data...")
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

# Optimize for faster loading during training
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# 3. Build the CNN Architecture
print("\nBuilding the Medical CNN...")
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(150, 150, 3)),
    tf.keras.layers.Rescaling(1./255),

    # Feature Extractors
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),

    # I added a 3rd Convolutional block here. X-rays are more complex than
    # cats/dogs, so we need a deeper network to find the subtle pneumonia patterns.
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),

    # The "Brain"
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5), # Prevents overfitting (memorizing the specific X-rays)
    tf.keras.layers.Dense(1, activation='sigmoid') # 0 = Normal, 1 = Pneumonia
])

# 4. Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# 5. Train it!
print("\nStarting Training (This will take a few minutes)...")
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=5)

# 6. SAVE THE MODEL!
model.save('pneumonia_model.keras')
print("\nModel successfully saved as 'pneumonia_model.keras'!")

Loading training data...
Found 5216 files belonging to 2 classes.

Loading validation data...
Found 624 files belonging to 2 classes.

Building the Medical CNN...

Starting Training (This will take a few minutes)...
Epoch 1/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 284s 2s/step - accuracy: 0.8969 - loss: 0.2579 - val_accuracy: 0.7420 - val_loss: 0.9121
Epoch 2/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 269s 1s/step - accuracy: 0.9546 - loss: 0.1207 - val_accuracy: 0.7404 - val_loss: 1.1175
Epoch 3/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 271s 1s/step - accuracy: 0.9659 - loss: 0.0950 - val_accuracy: 0.7981 - val_loss: 0.8044
Epoch 4/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 263s 1s/step - accuracy: 0.9768 - loss: 0.0678 - val_accuracy: 0.7869 - val_loss: 0.9276
Epoch 5/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 237s 1s/step - accuracy: 0.9780 - loss: 0.0669 - val_accuracy: 0.7708 - val_loss: 0.9660

Model successfully saved as 'pneumonia_model.keras'!


In [ ]:
# 1. Install Gradio (Colab doesn't have it by default)
!pip install -q gradio

import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image

# 2. Load the "Brain" you just saved
print("Loading model...")
model = tf.keras.models.load_model('pneumonia_model.keras')

# 3. Create the prediction function
def predict_pneumonia(img):
    if img is None:
        return "Please upload an image."

    # Resize the image to 150x150, which is what our model expects
    img = img.resize((150, 150))

    # Convert the image to an array of pixels
    img_array = np.array(img)

    # Our model expects a "batch" of images, so we add an extra dimension to make it a batch of 1
    img_array = np.expand_dims(img_array, axis=0)

    # Make the prediction!
    # Because we used a sigmoid activation, it outputs a single number between 0 and 1.
    # 'NORMAL' was folder 0, 'PNEUMONIA' was folder 1.
    prediction = model.predict(img_array)[0][0]

    if prediction >= 0.5:
        return f"🚨 Pneumonia Detected (Confidence: {prediction * 100:.2f}%)"
    else:
        return f"✅ Normal Lungs (Confidence: {(1 - prediction) * 100:.2f}%)"

# 4. Build the Web App UI
print("Launching Web App...")
interface = gr.Interface(
    fn=predict_pneumonia,           # The function to run
    inputs=gr.Image(type="pil"),    # The input: An image upload box
    outputs="text",                 # The output: Our text string
    title="Pneumonia Detection AI",
    description="Upload a Chest X-ray to predict if it shows signs of Pneumonia."
)

# 5. Launch it and generate a public link!
interface.launch(share=True, debug=True)

Loading model...
Launching Web App...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b94d8a4857c69d9f1a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
